# Hill Climbing (Ascenso de colina)

Busqueda **local**: a diferencia de DFS, BFS o A*, no mantiene una frontera ni
un conjunto de explorados, solo el **estado actual**. En cada paso se mueve al
vecino con mejor valor y se detiene cuando ningun vecino mejora.

- **Ventaja:** memoria constante, sirve en espacios de estados enormes.
- **Desventaja:** es incompleto, se atora en **maximos locales**, **mesetas** y
  **crestas**.

Reutilizamos las clases abstractas `Problem` y `Node` de las clases anteriores.

## 1. Clase abstracta `Problem`



In [2]:
# Clase abstracta
class Problem:
    def __init__(self, initial, goal=None):
        self.initial = initial # Estado inicial
        self.goal = goal # Meta

    def actions(self, state):
        raise NotImplementedError

    # Funcion de transicion
    def result(self, state, action):
        raise NotImplementedError

    # Funcion de desempeno
    def is_goal(self, state):
        return self.goal == state

    def action_cost(self, state1, action, state2):
        return 1

    def h(self, state):
        return 0

    # --- Agregados para busqueda local ---
    # Funcion objetivo: hill climbing MAXIMIZA este valor
    def value(self, state):
        return -self.h(state)

    # Estado aleatorio, usado por los reinicios aleatorios
    def random_state(self):
        raise NotImplementedError

## 2. Clase `Node`


In [3]:
class Node:
    def __init__(self, state, parent = None, action = None, path_cost = 0):
        self.state = state
        self.parent = parent
        self.action = action
        self.path_cost = path_cost

    def path(self):
        lista_path = []
        node = self
        while node:
            lista_path.append(node.state)
            node = node.parent
        return lista_path[::-1]

    def expand(self, problem):
        lista = []
        for action in problem.actions(self.state):
            lista.append(self.child_node(problem, action))
        return lista

    def child_node(self, problem, action):
        next_state = problem.result(self.state, action)
        step_cost = problem.action_cost(self.state, action, next_state)
        return Node(next_state, self, action, self.path_cost + step_cost)

## 3. El algoritmo

### 3.1 Elegir al mejor vecino


In [4]:
import random

def mejor_vecino(neighbors, problem):
    """El vecino con el mayor valor. Si hay empate, uno al azar."""
    mejor_valor = max(problem.value(node.state) for node in neighbors)
    empatados = [node for node in neighbors if problem.value(node.state) == mejor_valor]
    return random.choice(empatados)

### 3.2 Ascenso por maxima pendiente (*steepest-ascent*)


In [5]:
def hill_climbing(problem):
    """Ascenso de colina por maxima pendiente."""
    current = Node(problem.initial)

    while True:
        neighbors = current.expand(problem)
        if not neighbors:
            return current

        # El vecino "mas alto" de la colina
        neighbor = mejor_vecino(neighbors, problem)

        # Ningun vecino mejora: cima (maximo local o global)
        if problem.value(neighbor.state) <= problem.value(current.state):
            return current

        current = neighbor

### 3.3 Con movimientos laterales (*sideways moves*)


In [6]:
def hill_climbing_sideways(problem, max_sideways = 100):
    """Permite hasta max_sideways movimientos de igual valor para cruzar mesetas."""
    current = Node(problem.initial)
    sideways = 0

    while True:
        neighbors = current.expand(problem)
        if not neighbors:
            return current

        neighbor = mejor_vecino(neighbors, problem)
        valor_vecino = problem.value(neighbor.state)
        valor_actual = problem.value(current.state)

        if valor_vecino < valor_actual: # Empeora: nos quedamos
            return current

        if valor_vecino == valor_actual: # Meseta
            if sideways >= max_sideways:
                return current
            sideways += 1
        else: # Subimos de verdad
            sideways = 0

        current = neighbor

### 3.4 Con reinicios aleatorios (*random-restart*)


In [7]:
def random_restart_hill_climbing(problem, max_restarts = 100, search = hill_climbing):
    """Repite hill climbing desde estados aleatorios hasta hallar la meta."""
    inicial_original = problem.initial
    mejor = None
    reinicios = 0

    for reinicios in range(1, max_restarts + 1):
        problem.initial = problem.random_state()
        node = search(problem)

        if mejor is None or problem.value(node.state) > problem.value(mejor.state):
            mejor = node

        if problem.is_goal(node.state):
            break

    problem.initial = inicial_original # Dejamos el problema como estaba
    return mejor, reinicios

## 4. Mapa de Rumania


In [8]:
romania = {
    'Arad': {'Zerind': 75, 'Sibiu': 140, 'Timisoara': 118},
    'Zerind': {'Arad': 75, 'Oradea': 71},
    'Oradea': {'Zerind': 71, 'Sibiu': 151},
    'Sibiu': {'Arad': 140, 'Oradea': 151, 'Fagaras': 99, 'Rimnicu Vilcea': 80},
    'Timisoara': {'Arad': 118, 'Lugoj': 111},
    'Lugoj': {'Timisoara': 111, 'Mehadia': 70},
    'Mehadia': {'Lugoj': 70, 'Dobreta': 75},
    'Dobreta': {'Mehadia': 75, 'Craiova': 120},
    'Craiova': {'Dobreta': 120, 'Rimnicu Vilcea': 146, 'Pitesti': 138},
    'Rimnicu Vilcea': {'Sibiu': 80, 'Craiova': 146, 'Pitesti': 97},
    'Fagaras': {'Sibiu': 99, 'Bucarest': 211},
    'Pitesti': {'Rimnicu Vilcea': 97, 'Craiova': 138, 'Bucarest': 101},
    'Bucarest': {'Fagaras': 211, 'Pitesti': 101, 'Giurgiu': 90, 'Urziceni': 85},
    'Giurgiu': {'Bucarest': 90},
    'Urziceni': {'Bucarest': 85, 'Hirsova': 98, 'Vaslui': 142},
    'Hirsova': {'Urziceni': 98, 'Eforie': 86},
    'Eforie': {'Hirsova': 86},
    'Vaslui': {'Urziceni': 142, 'Iasi': 92},
    'Iasi': {'Vaslui': 92, 'Neamt': 87},
    'Neamt': {'Iasi': 87},
}

# Distancias lineales de cada ciudad a Bucarest
straight_line_distance = {
    'Arad': 366,
    'Bucarest': 0,
    'Craiova': 160,
    'Dobreta': 242,
    'Eforie': 161,
    'Fagaras': 178,
    'Giurgiu': 77,
    'Hirsova': 151,
    'Iasi': 226,
    'Lugoj': 244,
    'Mehadia': 241,
    'Neamt': 234,
    'Oradea': 380,
    'Pitesti': 98,
    'Rimnicu Vilcea': 193,
    'Sibiu': 253,
    'Timisoara': 329,
    'Urziceni': 80,
    'Vaslui': 199,
    'Zerind': 374,
}

In [9]:
class GraphHillClimbingProblem(Problem):
    def __init__(self, initial, goal, graph):
        super().__init__(initial, goal)
        self.graph = graph

    def actions(self, state):
        lista = []
        for key in self.graph[state].keys():
            lista.append(key)
        return lista

    # Funcion de transicion
    def result(self, state, action):
        return action

    def action_cost(self, state1, action, state2):
        return self.graph[state1][state2]

    def h(self, state):
        return straight_line_distance[state]

    # value() se hereda de Problem: value(state) = -h(state)

    def random_state(self):
        return random.choice(list(self.graph.keys()))

### 4.1 Caso exitoso: desde Arad

In [10]:
problem = GraphHillClimbingProblem("Arad", "Bucarest", romania)
node = hill_climbing(problem)

print("Trayectoria:", node.path())
print("Llego a la meta:", problem.is_goal(node.state))
print("Costo del camino:", node.path_cost)

Trayectoria: ['Arad', 'Sibiu', 'Fagaras', 'Bucarest']
Llego a la meta: True
Costo del camino: 450


Encuentra Bucarest con un costo de **450** (Arad - Sibiu - Fagaras - Bucarest),
pero el camino optimo que devuelve A* cuesta **418**
(Arad - Sibiu - Rimnicu Vilcea - Pitesti - Bucarest). Hill climbing **no
garantiza optimalidad**: desde Sibiu, Fagaras (h = 178) se ve mas cerca que
Rimnicu Vilcea (h = 193) y despues ya no hay forma de regresar.

### 4.2 Caso atorado: desde Lugoj

**Mehadia** es un maximo local: su heuristica es 241 y la de sus dos vecinos es
peor (Lugoj 244, Dobreta 242). El algoritmo llega ahi y no puede bajar para
despues volver a subir.

In [11]:
problem = GraphHillClimbingProblem("Lugoj", "Bucarest", romania)
node = hill_climbing(problem)

print("Trayectoria:", node.path())
print("Llego a la meta:", problem.is_goal(node.state))
print("Se atoro en:", node.state, "con h =", problem.h(node.state))
print("Vecinos:", {c: straight_line_distance[c] for c in romania[node.state]})

Trayectoria: ['Lugoj', 'Mehadia']
Llego a la meta: False
Se atoro en: Mehadia con h = 241
Vecinos: {'Lugoj': 244, 'Dobreta': 242}


### 4.3 Desde cada ciudad del mapa

In [12]:
exitos = 0
for ciudad in romania:
    problem = GraphHillClimbingProblem(ciudad, "Bucarest", romania)
    node = hill_climbing(problem)
    ok = problem.is_goal(node.state)
    exitos += ok
    print(f"{ciudad:16} -> {node.state:16} {'meta' if ok else 'ATORADO'} (costo {node.path_cost})")

print(f"\nExito en {exitos} de {len(romania)} ciudades")

Arad             -> Bucarest         meta (costo 450)
Zerind           -> Bucarest         meta (costo 525)
Oradea           -> Bucarest         meta (costo 461)
Sibiu            -> Bucarest         meta (costo 310)
Timisoara        -> Mehadia          ATORADO (costo 181)
Lugoj            -> Mehadia          ATORADO (costo 70)
Mehadia          -> Mehadia          ATORADO (costo 0)
Dobreta          -> Bucarest         meta (costo 359)
Craiova          -> Bucarest         meta (costo 239)
Rimnicu Vilcea   -> Bucarest         meta (costo 198)
Fagaras          -> Bucarest         meta (costo 211)
Pitesti          -> Bucarest         meta (costo 101)
Bucarest         -> Bucarest         meta (costo 0)
Giurgiu          -> Bucarest         meta (costo 90)
Urziceni         -> Bucarest         meta (costo 85)
Hirsova          -> Bucarest         meta (costo 183)
Eforie           -> Bucarest         meta (costo 269)
Vaslui           -> Bucarest         meta (costo 227)
Iasi             -> Bucare

## 5. Ejericio extra: las 8 reinas

El ejemplo clasico de busqueda local: colocar 8 reinas en un tablero de 8x8 sin
que se ataquen entre si.

- **Estado:** tupla de 8 numeros, `state[columna] = fila` de la reina de esa
  columna. Asi, por construccion, nunca hay dos reinas en la misma columna y el
  espacio se reduce de C(64,8) = 4,426,165,368 a 8^8 = 16,777,216 estados.
- **Acciones:** mover una reina a otra fila **de su misma columna**
  (8 x 7 = 56 vecinos).
- **Heuristica `h`:** numero de **pares** de reinas que se atacan.
- **Objetivo:** `value = -h`, que se maximiza; `h = 0` es la solucion.

In [13]:
class NQueensProblem(Problem):
    def __init__(self, initial = None, n = 8):
        self.n = n
        if initial is None:
            initial = tuple(random.randrange(n) for _ in range(n))
        super().__init__(initial, goal = None) # La meta no es un estado concreto

    # Mover una reina a otra fila dentro de su misma columna
    def actions(self, state):
        lista = []
        for columna in range(self.n):
            for fila in range(self.n):
                if state[columna] != fila:
                    lista.append((columna, fila))
        return lista

    # Funcion de transicion
    def result(self, state, action):
        columna, fila = action
        nuevo = list(state)
        nuevo[columna] = fila
        return tuple(nuevo)

    # Pares de reinas que se atacan (misma fila o misma diagonal)
    def h(self, state):
        ataques = 0
        for c1 in range(self.n):
            for c2 in range(c1 + 1, self.n):
                misma_fila = state[c1] == state[c2]
                misma_diagonal = abs(state[c1] - state[c2]) == abs(c1 - c2)
                if misma_fila or misma_diagonal:
                    ataques += 1
        return ataques

    # Funcion de desempeno: ninguna reina se ataca
    def is_goal(self, state):
        return self.h(state) == 0

    def random_state(self):
        return tuple(random.randrange(self.n) for _ in range(self.n))


def tablero(state):
    for fila in range(len(state)):
        print(" ".join("Q" if state[columna] == fila else "." for columna in range(len(state))))

### 5.1 Hill climbing simple

Desde un estado aleatorio se atora casi siempre: resuelve solo alrededor del
**15%** de los casos, aunque cuando lo logra es en muy pocos pasos (3 o 4).

In [14]:
random.seed(7) # Para que el resultado sea reproducible

problem = NQueensProblem()
print("Estado inicial:", problem.initial, "con", problem.h(problem.initial), "ataques")

node = hill_climbing(problem)
print("Estado final:  ", node.state, "con", problem.h(node.state), "ataques")
print("Resuelto:", problem.is_goal(node.state))
print("Pasos:", len(node.path()) - 1)
print()
tablero(node.state)

Estado inicial: (5, 2, 6, 0, 1, 1, 5, 0) con 6 ataques
Estado final:   (5, 2, 6, 1, 7, 1, 3, 0) con 1 ataques
Resuelto: False
Pasos: 4

. . . . . . . Q
. . . Q . Q . .
. Q . . . . . .
. . . . . . Q .
. . . . . . . .
Q . . . . . . .
. . Q . . . . .
. . . . Q . . .


### 5.2 Tasa de exito de cada variante

Comparamos las dos versiones sobre 200 estados iniciales aleatorios.

In [15]:
def tasa_de_exito(busqueda, intentos = 200, semilla = 0, **kwargs):
    random.seed(semilla)
    exitos = 0
    pasos = 0
    for _ in range(intentos):
        problem = NQueensProblem()
        node = busqueda(problem, **kwargs)
        exitos += problem.is_goal(node.state)
        pasos += len(node.path()) - 1
    return 100 * exitos / intentos, pasos / intentos


p, pasos = tasa_de_exito(hill_climbing)
print(f"Simple                   : {p:5.1f}% de exito, {pasos:.1f} pasos en promedio")

p, pasos = tasa_de_exito(hill_climbing_sideways, max_sideways = 100)
print(f"Con movimientos laterales: {p:5.1f}% de exito, {pasos:.1f} pasos en promedio")

Simple                   :  17.0% de exito, 3.1 pasos en promedio
Con movimientos laterales:  93.0% de exito, 21.5 pasos en promedio


Los movimientos laterales suben la tasa de exito de ~17% a ~93%, a costa de
usar muchos mas pasos (de 3 a mas de 20 en promedio). Funcionan tan bien porque
en las 8 reinas los estados donde el algoritmo se atora casi siempre son
**mesetas**, no maximos locales estrictos: hay por donde caminar de lado hasta
encontrar una salida hacia arriba.

### 5.3 Reinicios aleatorios

Siempre encuentra la solucion, solo hay que dejarlo reintentar. Si la
probabilidad de exito de un intento es *p*, el numero esperado de reinicios es
*1/p*: con *p* = 0.15 salen unos **7 reinicios**, justo lo que se observa abajo.

In [16]:
random.seed(1)

problem = NQueensProblem()
node, reinicios = random_restart_hill_climbing(problem, max_restarts = 100)

print("Solucion:", node.state)
print("Ataques:", problem.h(node.state))
print("Reinicios necesarios:", reinicios)
print()
tablero(node.state)

Solucion: (2, 5, 7, 0, 4, 6, 1, 3)
Ataques: 0
Reinicios necesarios: 7

. . . Q . . . .
. . . . . . Q .
Q . . . . . . .
. . . . . . . Q
. . . . Q . . .
. Q . . . . . .
. . . . . Q . .
. . Q . . . . .


In [17]:
# Numero de reinicios que necesita en promedio
random.seed(0)
total = 0
for _ in range(100):
    problem = NQueensProblem()
    node, reinicios = random_restart_hill_climbing(problem, max_restarts = 1000)
    total += reinicios

print(f"Reinicios promedio para resolver 8 reinas: {total / 100:.2f}")

Reinicios promedio para resolver 8 reinas: 7.41


## 6. Resumen

| Algoritmo | Completo | Optimo | Memoria |
|---|---|---|---|
| BFS | Si | Si (con costo uniforme) | O(b^d) |
| DFS | No | No | O(bm) |
| A* | Si | Si (con h admisible) | O(b^d) |
| Hill climbing | **No** | **No** | **O(1)** |
| Hill climbing con reinicios | Si (en el limite) | No | O(1) |

Hill climbing cambia garantias por memoria. Sirve cuando el espacio de estados
es demasiado grande para guardar una frontera y nos basta con una solucion
suficientemente buena. Sus tres enemigos son los **maximos locales**, las
**mesetas** y las **crestas**; se combaten con movimientos laterales, reinicios
aleatorios y, mas adelante, con **recocido simulado** (*simulated annealing*).